# Fine-tune GPT-3.5 Turbo for Customer Churn Prediction

This notebook guides you through fine-tuning a GPT-3.5 Turbo model using OpenAI's API.

## Prerequisites
- OpenAI API key with fine-tuning access
- Training dataset: `churn_dataset_gpt35_turbo.jsonl`
- Environment variable: `OPEN_AI_FINE_TUNING_KEY`

## 1. Setup and Initialization

In [ ]:
%pip install openai -q

In [ ]:
import os
import json
import time
from datetime import datetime
from openai import OpenAI
from IPython import get_ipython

# Load API key from environment
api_key = get_ipython().getoutput('echo $OPEN_AI_FINE_TUNING_KEY')
if api_key and api_key[0].strip():
    os.environ['OPEN_AI_FINE_TUNING_KEY'] = api_key[0].strip()
    print("✅ API key loaded")
else:
    raise ValueError("❌ OPEN_AI_FINE_TUNING_KEY environment variable not found")

client = OpenAI(api_key=os.getenv('OPEN_AI_FINE_TUNING_KEY'))
print("✅ OpenAI client initialized")

## 2. Prepare Training Data

In [ ]:
TRAINING_FILE = "churn_dataset_gpt35_turbo.jsonl"

if not os.path.exists(TRAINING_FILE):
    raise FileNotFoundError(f"❌ Training file not found: {TRAINING_FILE}")

# Inspect training data
print(f"📄 Training file: {TRAINING_FILE}")
with open(TRAINING_FILE, 'r', encoding='utf-8') as f:
    sample = json.loads(f.readline())
    print("\nSample record:")
    print(json.dumps(sample, indent=2))
    
    # Count total records
    f.seek(0)
    total_records = sum(1 for _ in f)
    print(f"\n✅ Total records: {total_records}")

## 3. Upload Training File

In [ ]:
print(f"📤 Uploading {TRAINING_FILE} to OpenAI...")

with open(TRAINING_FILE, 'rb') as f:
    training_file = client.files.create(file=f, purpose='fine-tune')

TRAINING_FILE_ID = training_file.id

print(f"✅ Upload complete")
print(f"   File ID: {TRAINING_FILE_ID}")
print(f"   Size: {training_file.bytes:,} bytes")

## 4. Create Fine-tuning Job

In [ ]:
print("🚀 Creating fine-tuning job...")

fine_tuning_job = client.fine_tuning.jobs.create(
    training_file=TRAINING_FILE_ID,
    model="gpt-3.5-turbo",
    hyperparameters={"n_epochs": 3},
    suffix="churn-prediction"
)

FINE_TUNING_JOB_ID = fine_tuning_job.id

print(f"✅ Job created")
print(f"   Job ID: {FINE_TUNING_JOB_ID}")
print(f"   Status: {fine_tuning_job.status}")
print(f"   Created: {datetime.fromtimestamp(fine_tuning_job.created_at)}")

## 5. Monitor Training Progress

In [ ]:
# Run this cell multiple times to check progress
job = client.fine_tuning.jobs.retrieve(FINE_TUNING_JOB_ID)

print(f"📊 Job Status: {job.status}")
print(f"   Job ID: {job.id}")

if hasattr(job, 'trained_tokens') and job.trained_tokens:
    print(f"   Trained Tokens: {job.trained_tokens:,}")

if job.status == 'succeeded':
    FINE_TUNED_MODEL = job.fine_tuned_model
    print(f"\n✅ Training complete!")
    print(f"   Model: {FINE_TUNED_MODEL}")
    print(f"\n💾 Save this model ID to .env as CUSTOMER_CHURN_OPEN_AI_MODEL")
elif job.status == 'failed':
    print(f"\n❌ Training failed")
    if hasattr(job, 'error'):
        print(f"   Error: {job.error}")
else:
    print(f"\n⏳ Training in progress... Run this cell again to check status.")

## 6. Training Metrics & Cost

In [ ]:
job = client.fine_tuning.jobs.retrieve(FINE_TUNING_JOB_ID)

if job.status == 'succeeded':
    print("📊 Training Summary:")
    print(f"   Model: {job.fine_tuned_model}")
    
    if hasattr(job, 'trained_tokens') and job.trained_tokens:
        trained_tokens = job.trained_tokens
        training_cost = (trained_tokens / 1_000_000) * 8.00  # $8/1M tokens
        print(f"   Trained Tokens: {trained_tokens:,}")
        print(f"   Estimated Cost: ${training_cost:.2f}")
    
    if hasattr(job, 'finished_at') and job.finished_at:
        duration = (job.finished_at - job.created_at) / 60
        print(f"   Duration: {duration:.1f} minutes")
    
    # Get training events
    try:
        events = client.fine_tuning.jobs.list_events(job.id, limit=10)
        if events.data:
            print("\n   Recent events:")
            for event in events.data[:5]:
                print(f"     • {event.message}")
    except:
        pass
else:
    print(f"⏳ Training not complete yet (status: {job.status})")

## 7. List All Fine-tuning Jobs

In [ ]:
jobs = client.fine_tuning.jobs.list(limit=10)

print("📋 Your Fine-tuning Jobs:\n")
for job in jobs.data:
    status_icon = "✅" if job.status == 'succeeded' else "⏳" if job.status == 'running' else "❌"
    print(f"{status_icon} {job.id}")
    print(f"   Status: {job.status}")
    if hasattr(job, 'fine_tuned_model') and job.fine_tuned_model:
        print(f"   Model: {job.fine_tuned_model}")
    print(f"   Created: {datetime.fromtimestamp(job.created_at).strftime('%Y-%m-%d %H:%M')}")
    print()